# Snakemake II
intermediate_data --> model_data


In [1]:
# 1. Setup, Dependencies, and Configuration
import sys
import os
import yaml
import glob

# Install Snakemake if not present
!{sys.executable} -m pip install snakemake -q

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define Core Paths
ROOT_DIR = '/content'
PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model'
modules_dir = os.path.join(PROJECT_DIR, 'modules')
python_scripts_dir = os.path.join(modules_dir, 'python_files')

os.makedirs(python_scripts_dir, exist_ok=True)
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# Convert Input Excel to YAML for Snakemake
excel_path = os.path.join(PROJECT_DIR, 'input_file.xlsx')
yaml_path = os.path.join(PROJECT_DIR, 'input_file.yaml')

if os.path.exists(excel_path):
    try:
        from modules.getting_input_data import get_input_data
        input_data = get_input_data(PROJECT_DIR, "input_file.xlsx")
        # Ensure we run all required scenarios and years

        input_data['scenario'] = ['GA', 'DE']
        input_data['year'] = ['2040', '2050']

        with open(yaml_path, 'w') as f:
            yaml.dump(input_data, f, default_flow_style=False)
        print("✅ Configuration loaded and saved to input_file.yaml")
    except Exception as e:
        print(f"❌ Error reading Excel: {e}")
else:
    print(f"❌ Missing {excel_path}! Please upload it.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 8.5 MB/s eta 0:00:00
Mounted at /content/drive
File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.
✅ Configuration loaded and saved to input_file.yaml


In [2]:
# 2. Convert Notebooks and Patch Scripts
print("--- 2a. Converting Notebooks ---")
nbs_to_convert = [
    "Intermediate_to_model_data.ipynb"
]
for nb_name in nbs_to_convert:
    nb_path = os.path.join(modules_dir, nb_name)
    if os.path.exists(nb_path):
        os.system(f"jupyter nbconvert --to python '{nb_path}' --output-dir='{python_scripts_dir}'")

print("--- 2b. Patching Scripts ---")
injection_code = f"""
# --- INJECTED BY PIPELINE ---
import yaml
import argparse
import sys
import os

parser = argparse.ArgumentParser()
parser.add_argument('--scenario', type=str, default='GA')
parser.add_argument('--year', type=str, default='2040')
args, unknown = parser.parse_known_args()

ROOT_DIR = '/content'
PROJECT_DIR = '{PROJECT_DIR}'

yaml_path = os.path.join(PROJECT_DIR, 'input_file.yaml')
with open(yaml_path, 'r') as f:
    input_data = yaml.safe_load(f)

input_data['scenario'] = args.scenario
input_data['year'] = int(args.year)  # Cast to integer for Pandas indexing

DATA_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data["raw_data_dir"])
output_dir = os.path.join(PROJECT_DIR, str(input_data["data_set"]), input_data['model_data_dir'], input_data["project_name"], input_data["scenario"], str(input_data["year"]))
MODEL_DATA_DIR = output_dir  # Fix for NameError
os.makedirs(output_dir, exist_ok=True)
# --------------------------\n
"""

import re
for py_file in glob.glob(os.path.join(python_scripts_dir, '*.py')):
    if 'getting_input_data' in py_file: continue

    with open(py_file, 'r') as f_read:
        lines = f_read.readlines()

    with open(py_file, 'w') as f_write:
        f_write.write(injection_code)
        for line in lines:
            if 'getting_input_data' in line or 'input_data = get_input_data' in line:
                f_write.write('# [BLOCKED EXCEL] ' + line)
            elif any(x in line for x in ['ROOT_DIR =', 'PROJECT_DIR =', 'DATA_DIR =', 'output_dir =', 'MODEL_DATA_DIR =']) and not line.strip().startswith('#'):
                f_write.write('# [BLOCKED PATH] ' + line)
            elif any(x in line for x in ['drive.mount', 'google.colab', 'get_ipython', '!pip', '%matplotlib']):
                f_write.write('# [REMOVED COLAB] ' + line)
            else:
                new_line = line.replace('display(', 'print(')

                # More robust regex replacement for target_year casting
                new_line = re.sub(r'\[\s*target_year\s*\]', '[int(target_year)]', new_line)
                new_line = re.sub(r'\[\[\s*target_year\s*\]\]', '[[int(target_year)]]', new_line)
                new_line = re.sub(r',\s*target_year\s*\]', ', int(target_year)]', new_line)

                # NEW: Replace 'Import_NTC_H2' with 'Import_NTC_elec' to fix NameError
                new_line = new_line.replace('Import_NTC_H2', 'Import_NTC_elec')

                # NEW: Handle NameError for NTC_H2_df when H2 is False
                if 'print(NTC_H2_df)' in new_line and not new_line.strip().startswith('#'):
                    leading_whitespace = new_line[:len(new_line) - len(new_line.lstrip())]
                    stripped_line_content = new_line.rstrip('\n').lstrip() # Get content without leading/trailing whitespace
                    f_write.write(f"{leading_whitespace}if input_data['H2']:\n")
                    f_write.write(f"{leading_whitespace}    {stripped_line_content}\n")
                    f_write.write(f"{leading_whitespace}else:\n")
                    f_write.write(f"{leading_whitespace}    print(\"NTC_H2_df not defined: H2 is False in config.\")\n")
                else:
                    f_write.write(new_line)

print("✅ All Python scripts successfully converted and patched for Snakemake.")

--- 2a. Converting Notebooks ---
--- 2b. Patching Scripts ---
✅ All Python scripts successfully converted and patched for Snakemake.


In [3]:
input_data

{'project_name': 'Europe',
 'regions': '["Albania", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Montenegro", "Netherlands", "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Serbia", "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "United Kingdom"]',
 'zones': '["AL00", "AT00", "BA00", "BE00", "BG00", "CH00", "CY00", "CZ00", "DE00", "DKE1", "DKW1", "EE00", "ES00", "FI00", "FR00", "GR00", "GR03", "HR00", "HU00", "IE00", "ITCA", "ITCN", "ITCS", "ITN1", "ITS1", "ITSA", "ITSI", "LT00", "LUB1", "LUF1", "LUG1", "LUV1", "LV00", "ME00", "SE02", "SE03", "SE04", "SI00", "SK00", "UK00", "UKNI", "MK00", "MT00", "NL00", "NOM1", "NON1", "NOS0", "PL00", "PT00", "RO00", "RS00", "SE01", "SE02", "SE03", "SE04", "SI00", "SK00", "UK00", "UKNI"]',
 'data_set': 'TYNDP_scenario_2024',
 'year': ['204

In [4]:
# 3. Generate Snakefile_II, Run Pipeline, and Verify

# --- 3a. Generating Snakefile_II ---
print("--- 3a. Generating Snakefile_II ---")
snakefile_content = (
    "configfile: 'input_file.yaml'\n\n"
    "rule all:\n"
    "    input:\n"
    "        expand(\"{data_set}/{model_data_dir}/{project_name}/{scenario}/{year}/model_data.json\",\n"
    "               data_set=config['data_set'],\n"
    "               model_data_dir=config['model_data_dir'],\n"
    "               scenario=config['scenario'],\n"
    "               year=config['year'],\n"
    "               project_name=config['project_name'])\n\n"
    "rule run_pipeline:\n"
    "    input:\n"
    f"        \"{input_data['input_data_path']}\"\n"
    "    output:\n"
    f"        \"{input_data['data_set']}/{input_data['model_data_dir']}/{input_data['project_name']}/{{scenario}}/{{year}}/model_data.json\"\n"
    "    shell:\n"
    "        \"\"\"\n"
    f"        python {python_scripts_dir}/Intermediate_to_model_data.py --scenario {{wildcards.scenario}} --year {{wildcards.year}}\n"
    "        \"\"\"\n"
)

with open(os.path.join(PROJECT_DIR, 'Snakefile_II'), 'w') as f:
    f.write(snakefile_content)

print("\n--- 3b. Running Snakemake Pipeline (Snakefile_II) ---")
os.chdir(PROJECT_DIR)

# Install missing dependencies for the Python script called by Snakemake
!pip install pypsa "xarray<=2024.9.0" -q

# Force run all combinations using Snakefile_II
!snakemake -s Snakefile_II -F --cores 1

print("\n--- 3c. Verifying Final Outputs ---")
# Re-define project_name and generated_files for verification
project_name = input_data['project_name']
output_pattern = os.path.join(PROJECT_DIR, f"{input_data['data_set']}/{input_data['model_data_dir']}/{project_name}/*/*/model_data.json")
generated_files = glob.glob(output_pattern)

if len(generated_files) == 4:
    print(f"🎉 SUCCESS! Found all {len(generated_files)} expected output files for {project_name}:")
else:
    print(f"⚠️ Partial/Failed run. Found {len(generated_files)} output files (expected 4) for {project_name}:")

for f in sorted(generated_files):
    print(f" - {f}")

--- 3a. Generating Snakefile_II ---

--- 3b. Running Snakemake Pipeline (Snakefile_II) ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.7/358.7 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.6/184.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.5 MB/s eta 0:00:00
Assuming unrestricted shared filesystem usage.

SNAKEMAKE
  Date: 2026-06-16 15:01:48
  Workflow ID: ffde552a-c1ca-407a-bc00-6c87258ca339
  Platform: Linux-6.6.122+-x86_64-with-glibc2.35
  Host: ce